# 00 — Carga, Tratamento e Validação Topológica (ego-Gplus / G2)

**Aluno:** Vitor Renato Santos dos Santos  
**Matrícula:** 222217109 — UFBA  
**Dataset:** ego-Gplus (Google+ social circles) — SNAP  
**Grafo atribuído:** G2 (calculado via $f(M) = (\sum d_i \times (u_2 + 1)) \bmod 129 = 26 \times 10 \bmod 129 = 2$)

---

### Objetivos deste Notebook:
1. Carregar o arquivo original `gplus_combined.txt.gz` (30.494.866 linhas) em streaming (evitando picos de memória).
2. Desduplicar arcos dirigidos (eliminando sobreposição de ego-networks).
3. Mapear os identificadores originais de 21 dígitos para inteiros contíguos $0 \dots N-1$.
4. Mensurar a reciprocidade de arcos dirigidos ($2 \times N_{\text{rec}} / N_{\text{arcos}}$).
5. Converter o grafo em simples e não-dirigido (colapsando pares mútuos em arestas únicas e descartando self-loops).
6. Extrair a Maior Componente Conexa (GCC).
7. Apresentar a **Tabela Antes/Depois** com métricas estruturais.
8. Persistir o grafo tratado em formato nativo `igraph` (`.pickle`) e formato matricial CSR (`.npz` com `indptr` e `indices`).

In [ ]:
import os
import sys
import time
import json
import gzip
import numpy as np
import pandas as pd
import scipy.sparse as sp
import igraph as ig

os.environ["MPLCONFIGDIR"] = os.path.abspath(".matplotlib_cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from src.graph_loader import load_and_process_gplus, save_processed_data, generate_directed_npz

## 1. Configuração de Parâmetros e Caminhos

Permitimos alternar entre **Modo Teste** (primeiras 1.000.000 de linhas) para validação rápida de código e consumo de memória, e **Modo Completo** (todas as ~30.5M linhas).

In [ ]:
RAW_DATA_PATH = "data/raw/gplus_combined.txt.gz"
if not os.path.exists(RAW_DATA_PATH):
    RAW_DATA_PATH = "gplus_combined.txt.gz"

PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

TEST_MODE = False if "--full" in sys.argv else True
MAX_LINES = 1_000_000 if TEST_MODE else None

print(f"Arquivo de entrada: {RAW_DATA_PATH}")
print(f"Modo de execução: {'TESTE (1.000.000 linhas)' if TEST_MODE else 'COMPLETO (30.494.866 linhas)'}")

## 2. Execução da Carga em Streaming e Pipeline de Tratamento

In [ ]:
t_start = time.perf_counter()

gcc, csr_gcc, stats, node_map = load_and_process_gplus(
    filepath=RAW_DATA_PATH,
    max_lines=MAX_LINES,
    verbose=True
)

elapsed_total = time.perf_counter() - t_start
print(f"\nPipeline concluído com sucesso em {elapsed_total:.2f} segundos!")

## 3. Tabela Comparativa: Antes vs. Depois do Tratamento

In [ ]:
antes = stats["antes"]
depois = stats["depois"]

tabela_dados = [
    ("Linhas brutas processadas", f"{antes['total_linhas_lidas']:,}", "—"),
    ("Linhas duplicadas (sobreposição)", f"{antes['linhas_duplicadas']:,}", "0 (removidas)"),
    ("Self-loops detectados", f"{antes['self_loops']:,}", "0 (removidos)"),
    ("Tipo do grafo", "Dígrafo (Dirigido)", "Grafo Simples Não-Dirigido"),
    ("Arcos dirigidos únicos", f"{antes['arcos_dirigidos_unicos']:,}", "—"),
    ("Pares recíprocos (a↔b)", f"{antes['pares_reciprocos']:,}", f"Colapsados em {depois['arestas_nao_dirigidas_gcc']:,} arestas"),
    ("Taxa de reciprocidade de arcos", f"{antes['taxa_reciprocidade_arcos_pct']:.2f}%", "Simétrico (100%)"),
    ("Vértices", f"{antes['vertices_unicos']:,}", f"{depois['vertices_gcc']:,} (GCC: {depois['fracao_gcc_pct']:.2f}%)"),
    ("Arestas não-dirigidas", "—", f"{depois['arestas_nao_dirigidas_gcc']:,}"),
    ("Componentes conexas", "—", f"{depois['num_componentes_conexas']:,}"),
    ("Densidade", "—", f"{depois['densidade']:.6e}"),
    ("Grau mín / médio / máx", "—", f"{depois['grau_minimo']} / {depois['grau_medio']:.1f} / {depois['grau_maximo']:,}"),
]

df_comparativo = pd.DataFrame(tabela_dados, columns=["Métrica / Característica", "Antes (Bruto / Dígrafo)", "Depois (Tratado / GCC)"])
print(df_comparativo.to_string(index=False))

## 4. Verificação de Integridade do Grafo Tratado

Validamos as propriedades fundamentais exigidas pelo trabalho:
- Grafo simples: sem self-loops e sem arestas múltiplas.
- Grafo conexo: composto exatamente por 1 componente conexa.
- Consistência da matriz CSR: dimensões e número de elementos não-nulos ($2 \times E$).

In [ ]:
is_simple = gcc.is_simple()
is_connected = gcc.is_connected()
csr_shape = csr_gcc.shape
csr_nnz = csr_gcc.nnz
expected_nnz = 2 * gcc.ecount()

print(f"O grafo tratado é simples? {is_simple}")
print(f"O grafo tratado é conexo? {is_connected}")
print(f"Dimensão da matriz CSR: {csr_shape}")
print(f"Não-nulos na CSR: {csr_nnz:,} (Esperado: {expected_nnz:,} = 2 × arestas)")

assert is_simple, "Erro: Grafo tratado contém laços ou multi-arestas!"
assert is_connected, "Erro: Grafo tratado não é estritamente conexo!"
assert csr_nnz == expected_nnz, "Erro: Inconsistência no número de elementos não-nulos da CSR!"
print("Todos os testes de integridade passaram com sucesso!")

## 5. Persistência dos Dados Tratados

Gravamos:
1. `gplus_undirected_gcc.pickle`: Grafo igraph para uso direto na Parte I e validações.
2. `gplus_undirected_gcc.npz`: Arrays CSR (`indptr`, `indices`) e vetor de nós para algoritmos em Numba.
3. `stats_antes_depois.json`: Registro quantitativo para inclusão no relatório LaTeX.

In [ ]:
prefix = "gplus_undirected_gcc_test" if TEST_MODE else "gplus_undirected_gcc"

saved_files = save_processed_data(
    gcc=gcc,
    csr_gcc=csr_gcc,
    stats=stats,
    node_map=node_map,
    output_dir=PROCESSED_DIR,
    prefix=prefix
)

print("Arquivos persistidos com sucesso:")
for k, path in saved_files.items():
    file_size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  [{k}] {path} ({file_size_mb:.2f} MB)")

### 5.1 Persistência do Dígrafo Original (gplus_directed.npz)

Para alimentar algoritmos direcionados (como Tarjan para SCCs na Parte II),
persistimos a matriz CSR do dígrafo original (13.673.453 arcos únicos),
garantindo estritamente o **mesmo mapeamento contíguo de IDs** (`node_ids`).

In [ ]:
directed_npz_path = os.path.join(PROCESSED_DIR, "gplus_directed.npz")
if not os.path.exists(directed_npz_path):
    print("Gerando representação CSR do dígrafo original...")
    rev_nodes = np.empty(len(node_map), dtype="U21")
    for raw_id, int_id in node_map.items():
        if int_id < len(rev_nodes):
            rev_nodes[int_id] = raw_id
    generate_directed_npz(raw_path=RAW_DATA_PATH, node_ids=rev_nodes, output_path=directed_npz_path)
else:
    print(f"Dígrafo original já existe em: {directed_npz_path}")

d_dir = np.load(directed_npz_path)
d_undir = np.load(saved_files["npz"], allow_pickle=True)

print(f"Arcos no dígrafo: {len(d_dir['indices']):,} (Esperado: 13.673.453)")
print(f"Mapeamento de nós idêntico? {np.array_equal(d_dir['node_ids'], d_undir['node_ids'])}")
assert len(d_dir['indices']) == 13_673_453, "Erro: Contagem de arcos dirigidos incorreta!"
assert np.array_equal(d_dir['node_ids'], d_undir['node_ids']), "Erro: Mapeamento de nós difere do grafo não-dirigido!"
print("Dígrafo original validado com sucesso!")

## 6. Conclusão e Próximos Passos

O pipeline de carga e tratamento está homologado.
- Com os arrays CSR disponíveis no formato `.npz`, as implementações em Numba (`@njit`) nos notebooks subsequentes (01_parte1_analise_estrutural e 02_parte2_algoritmos) podem ser carregadas instantaneamente em menos de 100 ms sem reprocessar o arquivo bruto de 380 MB.